# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](Mundo.png)


## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [12]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # TODO: completa a partir de la imagen
    
        self.start = (0,0)
        self.walls = {
            (0, 3),
            (1,1),
            (2,4),
            (4,2),
        }

        self.slippery_states = {
            (1,2),
            (2,1),
            (3,3),
        }

        self.terminal_states = {
            # (row, col): reward
            (0,5): +10,
            (4,5): -10,
            (2,2): +2,
        }

        self.danger_states = {
            # (row, col): -3
            (1,4): -3,
            (4,1): -3,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        # TODO
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls



    def states(self):
        # TODO
        return [
                (row, col)
                for row in range(self.height)
                for col in range(self.width)
                if self.is_valid_state((row, col))
            ]


    def is_terminal(self, state):
    
        return state in self.terminal_states

    def get_reward(self, state):
        # TODO: implementa R(s)
        # Un estado terminal es absorbente:
            # una vez allí, el agente permanece en el mismo estado.
            if self.is_terminal(state):
                return self.terminal_states[state]
            elif state in self.danger_states:
                return self.danger_states[state]
            elif state in self.walls:
                return 0
            else:
                return self.living_reward
    

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            # En estados resbalosos, hay una probabilidad del 60% de ir en la dirección deseada
            # y una probabilidad del 20% de ir en una dirección contraria (izquierda o derecha).
            probabilities =[
                (action, 0.6),
                ((action[1], -action[0]), 0.2),  # izquierda
                ((-action[1], action[0]), 0.2),  # derecha
            ]
        else:
            probabilities = [
                (action, 0.9),
                ((action[1], -action[0]), 0.05),  # izquierda
                ((-action[1], action[0]), 0.05),  # derecha
            ]    
        transition_probs = {}
        for direction, probability in probabilities:
            next_state = (state[0] + direction[0], state[1] + direction[1])
  
            if not self.is_valid_state(next_state):
              next_state = state

            transition_probs[next_state] = (
                transition_probs.get(next_state, 0.0) + probability
            )

    

        return list(transition_probs.items())

                




### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [13]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [16]:
def expected_next_value(grid, state, action, V):
    # TODO:
    # sum_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # Inicialización de Value Iteration:
    # V_0(s) = 0 para todos los estados.
    # Antes de iterar, todavía no hemos propagado ninguna recompensa.
    V = {state: 0.0 for state in grid.states()}
    deltas = []


    # Cada vuelta de este ciclo corresponde a una iteración k.
    # En cada iteración calcularemos V_{k+1} a partir de V_k.
    for iteration in range(max_iter):
        # Usamos una copia para hacer una actualización sincrónica:
        # todos los V_{k+1}(s) se calculan usando únicamente V_k.
        V_new = V.copy()
        # Este valor implementa el criterio de convergencia:
        # max_s |V_{k+1}(s) - V_k(s)|.
        biggest_change = 0.0


        # Aplicamos la ecuación de Bellman a cada estado s.
        for state in grid.states():
            # En un estado terminal ya no hay decisiones futuras.
            # Su valor queda fijado por su recompensa R(s).
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:

                # Esta línea corresponde a:
                #
                #   max_a Σ_{s'} T(s,a,s') V_k(s')
                #
                # Para cada acción a:
                # 1. se calculan los posibles estados siguientes s',
                # 2. se pondera V_k(s') por T(s,a,s'),
                # 3. se suman esos valores esperados,
                # 4. se elige la acción con mayor valor esperado.
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )


                # Ecuación de Bellman de optimalidad:
                #
                # V_{k+1}(s) = R(s)
                #                + gamma * max_a Σ_{s'} T(s,a,s') V_k(s')
                #
                # En el código:
                # grid.get_reward(state)  -> R(s)
                # grid.gamma              -> gamma
                # best_expected_value     -> max_a Σ T V_k
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )

            biggest_change = max(
                biggest_change,
            # Cambio local del estado:
            # |V_{k+1}(s) - V_k(s)|
                abs(V_new[state] - V[state])
            )


        # Terminamos la iteración:
        # V_{k+1} pasa a ser V_k para la siguiente vuelta.
        V = V_new
        deltas.append(biggest_change)


        # Si ningún estado cambia más que theta, consideramos
        # que Value Iteration ha convergido.
        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # Una vez tenemos V*(s), extraemos la política óptima.
    # La política guarda una acción por cada estado no terminal.
    policy = {}


    # Evaluamos cada estado de manera independiente.
    for state in grid.states():
        if grid.is_terminal(state):
            continue


        # Policy extraction:
        #
        # pi*(s) = argmax_a sum T(s,a,s') V*(s')
        #
        # Aquí max(..., key=...) no devuelve el valor máximo:
        # devuelve la ACCIÓN que produce ese máximo.
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(
                grid, state, action, V
            )
        )

    return policy

V_star, iterations = value_iteration(grid)
policy_star = extract_policy(grid, V_star)





## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [18]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # POLICY EVALUATION:
    # calculamos V^pi(s) manteniendo la política fija.
    #
    # V_{k+1}^pi(s) = R(s)
    #                    + gamma Σ_{s'} T(s,pi(s),s') V_k^pi(s')
    #
    # IMPORTANTE: aquí NO usamos max.
    # La acción ya viene determinada por policy[state].
    # Inicializamos V_0^pi(s)=0.
    # Luego propagaremos las recompensas siguiendo la política.
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        # Actualización sincrónica:
        # construimos V_{k+1} usando solamente V_k.
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:

                # pi(s): la política fija la acción.
                # A diferencia de Value Iteration, NO buscamos
                # todavía la mejor acción.
                action = policy[state]


                # Bellman para una política fija:
                # R(s) + gamma * valor esperado al seguir pi(s).
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma
                    * expected_next_value(grid, state, action, V)
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new

        # Si V^pi prácticamente no cambia, la evaluación
        # de esta política ha convergido.
        if biggest_change < threshold:
            break

    return V, iteration + 1    



def policy_improvement(grid, V):
    # POLICY IMPROVEMENT:
    # ahora sí preguntamos si existe una acción mejor.
    #
    # pi_new(s) = argmax_a Σ_{s'} T(s,a,s') V^pi(s')
    new_policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            continue


        # Aquí aparece el equivalente al 'max' de Value Iteration.
        # max(..., key=...) devuelve la ACCIÓN que maximiza
        # el valor esperado; por eso implementa argmax.
        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(
                grid, state, action, V
            )
        )

    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # TODO:
    # 1. política inicial arbitraria
    # 2. evaluación
    # 3. mejora
    # 4. repetir hasta estabilidad
    # 
    initial_action = (0, 1)
    policy = {
        state: initial_action
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []

    for iteration in range(max_iter):
        V, eval_iterations = policy_evaluation(
            grid, policy, threshold=threshold
        )

        new_policy = policy_improvement(grid, V)


        # Contamos en cuántos estados cambió la acción.
        # Si changed == 0, la política es estable.
        changed = sum(
            new_policy[state] != policy[state]
            for state in new_policy
        )

        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": eval_iterations,
            "changed_actions": changed,
        })

        policy = new_policy

        # Política estable:
        # mejorarla ya no produce ninguna acción diferente.
        # Hemos alcanzado la política óptima.
        if changed == 0:
            break

    V, _ = policy_evaluation(
        grid, policy, threshold=threshold
    )

    return policy, V, history



## Parte 4 — Visualización y comparación


In [19]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [20]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 19

Valores:
 -2.574 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.185 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.220 |  -0.054 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.736 |  -0.690 |  +0.599 |  +0.231 |  +2.252 |  +3.870
 -2.701 |  -3.854 |   WALL   |  -0.748 |  +0.340 | -10.000

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  →  |  →  |  ↑ 
 ↑  |  ↑  |  #  |  →  |  ↑  | -10

=== POLICY ITERATION ===
Historia: [{'policy_iteration': 1, 'evaluation_sweeps': 76, 'changed_actions': 18}, {'policy_iteration': 2, 'evaluation_sweeps': 17, 'changed_actions': 7}, {'policy_iteration': 3, 'evaluation_sweeps': 19, 'changed_actions': 1}, {'policy_iteration': 4, 'evaluation_sweeps': 19, 'changed_actions': 0}]

Valores:
 -2.574 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.185 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.220 |  -0.054 |  +


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. **Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?**\
R/ Prefiere la entrega +10 pues los V(s) de los estados que se dirigen a +10 tienen mayores valores que los que se dirigen a +2.

2. **¿Por qué una recompensa menor podría ser óptima?**\
R/ Porque el algoritmo no compara solo las recompensas terminales, sino el retorno esperado completo. Una recompensa menor puede ser óptima si está más cerca, requiere menos pasos o presenta menos riesgo.

3. **¿En qué estados el piso resbaloso cambia la decisión?**\
R/ 
4. **¿Qué papel cumple el costo por paso `-1`?**\
R/ Que el algortimo no prefiera quedarse dando vueltas a ir a estados terminales.

5. **¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?**\
R/ Porque las probabilidades cambian dependiendo del tipo de estado. 

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.
